In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from collections import Counter

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

try:
    from imblearn.over_sampling import SMOTE, RandomOverSampler
    IMBLEARN_AVAILABLE = True
    print(" imblearn library loaded successfully!")
except ImportError:
    print(" imblearn not installed. Install with: pip install imbalanced-learn")
    IMBLEARN_AVAILABLE = False

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
    nltk.download('stopwords')

print(" All base libraries imported successfully!")

dataset_path = r"C:\Users\fiaaa\Desktop\phishing_detection_system\datasets\Phishing_Email.csv"

df = pd.read_csv(dataset_path)
print(f" Dataset loaded successfully!")
print(f" Dataset shape: {df.shape}")
print(f"\nFirst 5 rows:")
print(df.head())

print("\n Column names:")
print(df.columns.tolist())

print("\n Missing values:")
print(df.isnull().sum())

print("\n Class distribution (BEFORE balancing):")
print(df['Email Type'].value_counts())

if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)
    print("\n Dropped 'Unnamed: 0' column")

df['Email Text'] = df['Email Text'].fillna('')
print(f" Missing values filled: {df['Email Text'].isnull().sum()} remaining")

df['Email Type'] = df['Email Type'].str.strip()

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    try:
        tokens = word_tokenize(text)
    except:
        tokens = text.split()
    
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    tokens = [stemmer.stem(t) for t in tokens]
    
    return ' '.join(tokens)

test_text = "URGENT: Your account has been suspended! Please verify immediately."
processed = preprocess_text(test_text)
print(f"\nOriginal: {test_text}")
print(f"Processed: {processed}")

print("\n Preprocessing all emails...")
df['processed_text'] = df['Email Text'].apply(preprocess_text)
print(" Preprocessing complete!")

X = df['processed_text']
y = df['Email Type'].map({'Safe Email': 0, 'Phishing Email': 1})

print(f"\n Features shape: {X.shape}")
print(f" Labels shape: {y.shape}")
print(f"\nLabel distribution BEFORE balancing:")
print(f"Safe (0): {sum(y == 0)}")
print(f"Phishing (1): {sum(y == 1)}")

# %% [markdown]
# ## 8. Split Data

# %%
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print(f"\n Training set: {len(X_train)} samples")
print(f" Test set: {len(X_test)} samples")

# %% [markdown]
# ## 9. TF-IDF Vectorization

# %%
vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.8,
    ngram_range=(1, 2)
)

print(" Creating TF-IDF features...")
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

print(f"\n Training features shape: {X_train_vectorized.shape[0]} samples, {X_train_vectorized.shape[1]} features")
print(f" Test features shape: {X_test_vectorized.shape[0]} samples, {X_test_vectorized.shape[1]} features")

# %% [markdown]
# ## 10. Check Imbalance

# %%
print("\n Class distribution BEFORE balancing:")
print(f"Training set: {Counter(y_train)}")
print(f"Safe: {sum(y_train == 0)}, Phishing: {sum(y_train == 1)}")

# %% [markdown]
# ## 11. Apply SMOTE (if available)

# %%
if IMBLEARN_AVAILABLE:
    print("\n Applying SMOTE to balance training data...")
    
    # Initialize SMOTE
    smote = SMOTE(random_state=42)
    
    # Apply SMOTE to training features
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train_vectorized, y_train)
    
    #  FIXED: Use shape[0] instead of len() for sparse matrix
    print(f"\n Training features AFTER SMOTE: {X_train_balanced.shape[0]} samples, {X_train_balanced.shape[1]} features")
    print(f" Training labels AFTER SMOTE: {len(y_train_balanced)} samples")
    print(f"\n Class distribution AFTER SMOTE:")
    print(f"Safe: {sum(y_train_balanced == 0)}")
    print(f"Phishing: {sum(y_train_balanced == 1)}")
    
    use_smote = True
else:
    print("\n SMOTE not available. Using original data with class_weight='balanced'")
    X_train_balanced = X_train_vectorized
    y_train_balanced = y_train
    use_smote = False

# %% [markdown]
# ## 12. Train Random Forest

# %%
print("\n Training Random Forest model...")

if use_smote:
    # If SMOTE applied, we can use normal class_weight
    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        random_state=42,
        class_weight=None,  # SMOTE already balanced the data
        n_jobs=-1
    )
else:
    # If no SMOTE, use balanced class weights
    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    )

rf_model.fit(X_train_balanced, y_train_balanced)
print(" Model training complete!")

# %% [markdown]
# ## 13. Evaluate Model

# %%
y_pred = rf_model.predict(X_test_vectorized)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n Model Accuracy: {accuracy:.2%}")

print("\n Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Safe', 'Phishing']))

print("\n Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Calculate metrics
tn, fp, fn, tp = cm.ravel()
print(f"\n Detailed Metrics:")
print(f"True Positives (Correct Phishing): {tp}")
print(f"True Negatives (Correct Safe): {tn}")
print(f"False Positives (Safe as Phishing): {fp}")
print(f"False Negatives (Missed Phishing): {fn}")

# %% [markdown]
# ## 14. Feature Importance

# %%
feature_names = vectorizer.get_feature_names_out()
importances = rf_model.feature_importances_
top_indices = np.argsort(importances)[-20:][::-1]

print("\n🔝 Top 20 Important Features:")
for i, idx in enumerate(top_indices[:20]):
    print(f"{i+1:2d}. {feature_names[idx]:20s} → {importances[idx]:.4f}")

# %% [markdown]
# ## 15. Test with Sample Emails

# %%
test_emails = [
    "URGENT: Your account has been suspended! Click here to verify.",
    "Hi, how are you? Let's meet for coffee tomorrow.",
    "software at incredibly low prices (86% lower)...",
    "Your PayPal account needs verification.",
    "Meeting scheduled for tomorrow at 10 AM.",
    "Congratulations! You've won $1000. Click to claim.",
    "Your invoice #INV-2024-001 is attached.",
    "Security alert: New login from unknown device."
]

print("\n Testing Model:")
print("="*60)

for i, email in enumerate(test_emails, 1):
    processed = preprocess_text(email)
    features = vectorizer.transform([processed])
    pred = rf_model.predict(features)[0]
    proba = rf_model.predict_proba(features)[0]
    
    result = "PHISHING" if pred == 1 else "SAFE"
    confidence = max(proba) * 100
    
    # Get top features for this prediction
    if hasattr(features, 'toarray'):
        feature_values = features.toarray()[0]
    else:
        feature_values = features[0]
    
    weighted = importances * feature_values
    top_indices = np.argsort(weighted)[-3:][::-1]
    top_features = [feature_names[idx] for idx in top_indices if weighted[idx] > 0.01]
    
    print(f"\n{i}. Email: {email[:50]}...")
    print(f"   Result: {result}")
    print(f"   Confidence: {confidence:.1f}%")
    if top_features:
        print(f"   Key indicators: {', '.join(top_features)}")

# %% [markdown]
# ## 16. Save Model and Vectorizer

# %%
os.makedirs('../backend/models', exist_ok=True)

joblib.dump(rf_model, '../backend/models/rf_model.pkl')
joblib.dump(vectorizer, '../backend/models/tfidf.pkl')

print("\n Model saved to: ../backend/models/rf_model.pkl")
print(" Vectorizer saved to: ../backend/models/tfidf.pkl")

if os.path.exists('../backend/models/rf_model.pkl'):
    size = os.path.getsize('../backend/models/rf_model.pkl') / 1024 / 1024
    print(f" Model file size: {size:.2f} MB")

# %% [markdown]
# ## 17. Load Test

# %%
print("\n Testing model loading...")
loaded_model = joblib.load('../backend/models/rf_model.pkl')
loaded_vectorizer = joblib.load('../backend/models/tfidf.pkl')

test_email = "URGENT: Verify your account immediately!"
processed = preprocess_text(test_email)
features = loaded_vectorizer.transform([processed])
pred = loaded_model.predict(features)[0]

print(f"Model loaded successfully!")
print(f"Test prediction: {'PHISHING' if pred == 1 else 'SAFE'}")

# %% [markdown]
# ## Training Complete!

# %%
print("\n" + "="*50)
print("MODEL TRAINING COMPLETE!")
print("="*50)
print(f"\nSummary:")
print(f"- SMOTE used: {use_smote}")
print(f"- Original training samples: {len(X_train)}")
if use_smote:
    # FIXED: Use shape[0] for sparse matrix
    print(f"- After SMOTE: {X_train_balanced.shape[0]} samples")
print(f"- Accuracy: {accuracy:.2%}")
print(f"- Model saved to backend/models/")
print("\nRun backend: cd ../backend && python main.py")

✅ imblearn library loaded successfully!
✅ All base libraries imported successfully!
✅ Dataset loaded successfully!
📊 Dataset shape: (18650, 3)

First 5 rows:
   Unnamed: 0                                         Email Text  \
0           0  re : 6 . 1100 , disc : uniformitarianism , re ...   
1           1  the other side of * galicismos * * galicismo *...   
2           2  re : equistar deal tickets are you still avail...   
3           3  \nHello I am your hot lil horny toy.\n    I am...   
4           4  software at incredibly low prices ( 86 % lower...   

       Email Type  
0      Safe Email  
1      Safe Email  
2      Safe Email  
3  Phishing Email  
4  Phishing Email  

📋 Column names:
['Unnamed: 0', 'Email Text', 'Email Type']

🔍 Missing values:
Unnamed: 0     0
Email Text    16
Email Type     0
dtype: int64

📊 Class distribution (BEFORE balancing):
Email Type
Safe Email        11322
Phishing Email     7328
Name: count, dtype: int64

✅ Dropped 'Unnamed: 0' column
✅ Missing va

In [2]:
# Run this in a new cell
!pip install imbalanced-learn